# PA-SFT: Qwen3.5-4B on PA-SFT_dataset (ANSWER-LAST) — pip-installed LlamaFactory (Colab)

## 1. Install LlamaFactory via pip (+ transformers 5.x for Qwen3.5)

In [ ]:
!pip install -q "llamafactory[metrics] @ git+https://github.com/hiyouga/LLaMA-Factory.git"

!pip install -q --upgrade \
  "transformers>=5.2,<=5.6" \
  "tokenizers>=0.22" \
  "accelerate>=1.3,<=1.11" \
  "peft>=0.18,<=0.18.1" \
  "huggingface_hub>=0.27" \
  "datasets>=2.16,<=4.0" \
  "trl>=0.18,<=0.24" \
  bitsandbytes sentencepiece einops pillow

!pip check 2>/dev/null | grep -i "llamafactory" || echo "llamafactory deps OK"

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN   = userdata.get('HF_TOKEN')
HF_REPO_ID = 'minsu0567/IAD-X1-SFT-answer-last'
login(token=HF_TOKEN)
print('HuggingFace login OK.')

## 3. GPU check

In [ ]:
!nvidia-smi

## 4. Paths

In [ ]:
import os

DRIVE_ROOT     = '/content/drive/MyDrive'
IAD_R1_DIR     = f'{DRIVE_ROOT}/IAD-R1-main'
IAD_X1_DIR     = f'{DRIVE_ROOT}/IAD-X1'
SFT_SRC        = f'{IAD_X1_DIR}/sft_src'
DATASET_DIRS   = [
    f'{DRIVE_ROOT}/PA-SFT_dataset_2',
]
REORDERED_JSON = f'{DRIVE_ROOT}/merged_reordered_answer_last.json'
OUTPUT_DIR     = '/content/PA-SFT_output'

for p in [IAD_R1_DIR, SFT_SRC, *DATASET_DIRS]:
    assert os.path.isdir(p), f'Missing directory: {p}'
assert os.path.isfile(REORDERED_JSON), f'Missing: {REORDERED_JSON}'
assert os.path.isfile(f'{IAD_R1_DIR}/data/dataset_info.json')
assert os.path.isfile(f'{SFT_SRC}/pa_sft_train3.py')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('All paths OK.')
print('Output dir  :', OUTPUT_DIR)

## 5. Confirm dataset_info.json contains `PA_SFT_merged_2`

In [ ]:
import json
with open(f'{IAD_R1_DIR}/data/dataset_info.json', encoding='utf-8') as f:
    info = json.load(f)
assert 'PA_SFT_merged_2' in info, 'PA_SFT_merged_2 entry not found in dataset_info.json'
print('PA_SFT_merged_2 ->', info['PA_SFT_merged_2']['file_name'])

## 6. Register answer-last dataset

In [ ]:
import os, json

assert os.path.isfile(REORDERED_JSON), (
    f'Missing answer-last dataset: {REORDERED_JSON}\n'
    'Upload merged_reordered_answer_last.json to your Drive MyDrive root first.'
)

_info_path = f'{IAD_R1_DIR}/data/dataset_info.json'
with open(_info_path, encoding='utf-8') as f:
    _info = json.load(f)

assert 'PA_SFT_merged_2' in _info, 'PA_SFT_merged_2 base entry not found.'
_new = dict(_info['PA_SFT_merged_2'])
_new['file_name'] = REORDERED_JSON
_info['PA_SFT_2_answer_last'] = _new

with open(_info_path, 'w', encoding='utf-8') as f:
    json.dump(_info, f, ensure_ascii=False, indent=2)

print('Registered PA_SFT_2_answer_last ->', _info['PA_SFT_2_answer_last']['file_name'])

## 7. Sanity check — official `qwen3_5` template is present in the pip package

In [ ]:
from llamafactory.data.template import TEMPLATES

for name in ('qwen3_5', 'qwen3_5_nothink'):
    assert name in TEMPLATES, (
        f'{name} template not found in the installed LlamaFactory.\n'
        'Your version predates qwen3_5 support — re-run step 1 to install from GitHub main:\n'
        '  pip install "llamafactory[metrics] @ git+https://github.com/hiyouga/LLaMA-Factory.git"'
    )
    tpl = TEMPLATES[name]
    print(f'{name:16s} -> class={type(tpl).__name__:20s} '
          f'plugin={type(tpl.mm_plugin).__name__:16s} replace_eos={getattr(tpl, "replace_eos", None)}')

assert type(TEMPLATES['qwen3_5_nothink']).__name__ == 'Template', (
    'qwen3_5_nothink is unexpectedly a ReasoningTemplate — check upstream changes.'
)
print('\nUsing --template qwen3_5_nothink for training (no empty-think injection).')

import transformers
print('transformers   :', transformers.__version__)

## 8. Full PA-SFT training

In [ ]:
import os
os.environ['WANDB_MODE'] = 'offline' # 웹 전송 x (wandb 는 전송받은 로그 기록을 통해 웹 대시보드에 loss 그래프 그려주는 외부 서비스)
os.environ['DISABLE_VERSION_CHECK'] = '1' # llamafactory 의존성 검증 건너뜀.
os.environ.pop('PYTHONPATH', None)

cmd = f'''DISABLE_VERSION_CHECK=1 \
python {SFT_SRC}/pa_sft_train3.py \
  --stage sft \
  --do_train \
  --model_name_or_path Qwen/Qwen3.5-4B \
  --dataset PA_SFT_2_answer_last \
  --dataset_dir {IAD_R1_DIR}/data \
  --media_dir /content \
  --template qwen3_5_nothink \
  --finetuning_type full \
  --pure_bf16 \
  --optim adamw_bnb_8bit \
  --output_dir {OUTPUT_DIR} \
  --overwrite_cache \
  --gradient_checkpointing true \
  --warmup_steps 100 \
  --weight_decay 0.1 \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 1 \
  --learning_rate 1e-5 \
  --lr_scheduler_type cosine \
  --logging_steps 5 \
  --cutoff_len 8192 \
  --save_steps 500 \
  --save_total_limit 2 \
  --plot_loss \
  --num_train_epochs 1 \
  --bf16 2>&1 | tee {OUTPUT_DIR}/train.log'''

print(cmd)
!{cmd}

## 9. Push trained model to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi, create_repo

create_repo(HF_REPO_ID, repo_type='model', exist_ok=True, private=False, token=HF_TOKEN)
api = HfApi()
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=HF_REPO_ID,
    repo_type='model',
    token=HF_TOKEN,
)
print(f'Model pushed to https://huggingface.co/{HF_REPO_ID}')

## 10. Verify final checkpoint

In [ ]:
!ls -lh {OUTPUT_DIR}